In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.calibration import calibration_curve

In [2]:
DATA_DIR = 'data/processed/'

# Load predictions and uncertainty from ensemble
guide_mean_probs = np.load(DATA_DIR + 'guide_mean_probs.npy')
guide_uncertainty = np.load(DATA_DIR + 'guide_uncertainty.npy')
change_mean_probs = np.load(DATA_DIR + 'change_mean_probs.npy')
change_uncertainty = np.load(DATA_DIR + 'change_uncertainty.npy')

# Load true labels
y_guide_test = np.load(DATA_DIR + 'y_guide_test.npy')
y_change_test = np.load(DATA_DIR + 'y_change_test.npy')

print(f'GUIDE-seq test: {len(y_guide_test)} samples, {y_guide_test.sum():.0f} positives')
print(f'CHANGE-seq test: {len(y_change_test)} samples, {y_change_test.sum():.0f} positives')

GUIDE-seq test: 248631 samples, 107 positives
CHANGE-seq test: 520991 samples, 6610 positives


## Performance Metrics

In [3]:
# AUROC and AUPRC for both datasets
print('GUIDE-seq (in-distribution):')
print(f'  AUROC: {roc_auc_score(y_guide_test, guide_mean_probs):.4f}')
print(f'  AUPRC: {average_precision_score(y_guide_test, guide_mean_probs):.4f}')
print(f'  Brier Score: {brier_score_loss(y_guide_test, guide_mean_probs):.4f}')

print('\nCHANGE-seq (out-of-distribution):')
print(f'  AUROC: {roc_auc_score(y_change_test, change_mean_probs):.4f}')
print(f'  AUPRC: {average_precision_score(y_change_test, change_mean_probs):.4f}')
print(f'  Brier Score: {brier_score_loss(y_change_test, change_mean_probs):.4f}')

GUIDE-seq (in-distribution):
  AUROC: 0.8285
  AUPRC: 0.0098
  Brier Score: 0.0039

CHANGE-seq (out-of-distribution):
  AUROC: 0.8519
  AUPRC: 0.0947
  Brier Score: 0.0137


### Results

The ensemble achieves AUROC of 0.83 on GUIDE-seq (in-distribution) and 0.85 on CHANGE-seq (out-of-distribution), outperforming all three baselines on both datasets. AUPRC is low for GUIDE-seq (0.0098) due to the extreme class imbalance (~99.96% negatives), but stronger on CHANGE-seq (0.0947) where positives are less rare. Brier scores are low across both datasets but should be interpreted cautiously under severe class imbalance, as a naive zero-predictor would also achieve a low score.

## ECE and Reliability Diagrams